### Chi-Square Test of independence; are the two varaibles independent?

In [87]:
import pandas as pd
import scipy.stats as stats

### 1. Load the raw data 

In [88]:
df = pd.read_csv("./overwatch_players.csv", encoding="latin-1")

### 2. Inspect before doing anything

In [89]:
print(df.head(20))
print("\nTotal players:", len(df))

    player_id     role favorite_hero
0           1  Support         Moira
1           2      DPS         Genji
2           3     Tank         Moira
3           4      DPS         Genji
4           5  Support         Genji
5           6     Tank         Moira
6           7      DPS         Genji
7           8      DPS         Moira
8           9     Tank         Mauga
9          10      DPS         Genji
10         11      DPS         Genji
11         12      DPS         Genji
12         13     Tank         Genji
13         14  Support         Genji
14         15     Tank         Mauga
15         16      DPS         Genji
16         17  Support         Genji
17         18  Support         Moira
18         19      DPS         Mauga
19         20     Tank         Genji

Total players: 150


### 3. CLEAN the category labels (catch inconsistent spelling)

In [90]:
# 3.1 value_counts() lists each distinct label and how often it appears
print(f"{df['role'].value_counts()}")
print(f"\n {df['favorite_hero'].value_counts()}")


role
Support    55
DPS        50
Tank       45
Name: count, dtype: int64

 favorite_hero
Genji    60
Moira    50
Mauga    40
Name: count, dtype: int64


In [101]:
# 3.2 str.strip() removes stray spaces, .str.title() makes case consistent
# text.strip() is similar to string.strip() but applies to entire column series at once
df['role'] = df['role'].str.strip().str.title()
df['favorite_hero'] = df['favorite_hero'].str.strip().str.title()


### 4. BUILD the contingency table by COUNTING


In [92]:
# crosstab counts how many rows fall in each (role, hero) combination
contingency_table = pd.crosstab(df['role'], df['favorite_hero'])
print(contingency_table)

favorite_hero  Genji  Mauga  Moira
role                              
Dps               40      5      5
Support           10      5     40
Tank              10     30      5


### 5. Run the chi-square test

In [93]:
# chi2_contingency returns 4 things: the statistic, the p-value, degrees of freedom, expected table
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table, correction=False) 

### 6. Show expected coutns (the "if they were unrelated" world)

In [94]:
expected_df = pd.DataFrame(expected, index = contingency_table.index, columns = contingency_table.columns)
print(expected_df.round(2))

favorite_hero  Genji  Mauga  Moira
role                              
Dps             20.0  13.33  16.67
Support         22.0  14.67  18.33
Tank            18.0  12.00  15.00


### 7. Check the assumption before trusting the result

In [95]:
# every expected count should be at least 5
if (expected >=5).all():
  print("Assumption Okay; all counts greater than 5.")
else:
  print("WARNING! some expected counts may be less than 5.")

Assumption Okay; all counts greater than 5.


### 8. Report the numbers

In [96]:
# 8. REPORT the numbers
print(f"Chi-square = {chi2:.2f}")
print(f"Degrees of freedom = {dof}")
print(f"p-value = {p_value:.2e}")

Chi-square = 109.12
Degrees of freedom = 4
p-value = 1.12e-22


### 9a. Decide using critical value

In [97]:
alpha = 0.05

# use chi ppf that finds the chi value for this df and this alpha

critical_value = stats.chi2.ppf(1 - alpha, dof)
print(f"Critical value (alpha = {alpha} and degree of freedom = {dof}) is: {critical_value:.2f}")
print(F"Our Chi-square= {chi2}")

# Decision
if chi2 > critical_value:
  print(f"Since {chi2:.2f} > {critical_value:.2f}, Reject independence (reject the null hypothesis) -> role and hero ARE related")
else:
  print(f"Since {chi2:.2f} <= {critical_value:.2f}, Cannot reject")

Critical value (alpha = 0.05 and degree of freedom = 4) is: 9.49
Our Chi-square= 109.11994949494951
Since 109.12 > 9.49, Reject independence (reject the null hypothesis) -> role and hero ARE related


### 9b. Decide using p-value

In [98]:
# Decide using P-value;
if p_value < alpha:
    print(f"p < {alpha}: REJECT independence -> role and hero ARE related")
else:
    print(f"p >= {alpha}: cannot reject -> no evidence they are related")

p < 0.05: REJECT independence -> role and hero ARE related


### In-class activity
-  TASK 1 (code): Add normalize='index' to see each role's percentages across heroes. Run this line:

In [99]:
# your code, use the crosstab() function and in it use the normalize parameter to get there.
# print((pd.crosstab(df['role'], df['favorite_hero'], normalize='index') * 100).round(1))


- TASK 2 (write your answer as a comment):

-    a) Look at the Tank row. What % of Tanks favor Mauga?


your answer

-   b) In one sentence: are role and favorite hero related? How do you know?


your answer

-   c) In one sentence: why can we NOT say that playing Tank CAUSES someone to love Mauga?

your answer

### STRETCH (if you finish early):
####   Which single role-hero combination contributes MOST to the
####   chi-square? Run this and read the biggest number:

In [100]:
# chi2, p, dof, expected = stats.chi2_contingency(
#     pd.crosstab(df['role'], df['favorite_hero']), correction=False)
# contributions = (pd.crosstab(df['role'], df['favorite_hero']) - expected)**2 / expected
# print(contributions.round(2))